# CPBL 主客場勝率預測 — 完整 Python pipeline（單一自含 notebook）

**這一個 notebook 就是全部流程。** 不 clone repo、不跑 subprocess、不依賴
任何 `.py` 檔 —— step1→step1b→step2→step3 全部內嵌為下方 cell，
`Runtime → Run all` 由上往下跑完即產出所有結果與圖。

| Cell | 內容 |
|---|---|
| 2 | 套件 + 路徑 + 內嵌球場經緯度 lookup |
| 3 | 抓 rebas 2024+2023 資料（`USE_2023` 預設 True）|
| 4 | 投手資料診斷（schema + 先發 sanity）|
| 5 | **step1** 解析 rebas → `raw_games.csv`（含投手聚合 + 先發）|
| 6 | **step1b** Open-Meteo 天氣 → `games_with_weather.csv` |
| 7 | **step2** 特徵工程（Elo/PF/打者狀態/**投手滾動**）→ `model_ready_data.csv` |
| 8 | **step3** m1–m7 消融 + per-group season-OOF + 演算法/調參 + 校準 + Shiny artifacts |
| 9 | 結果：印 `_final_metrics.json`（含 `ablation_season_oof`）+ 4 張圖 |
| 10 | （選）打包 / 推回 artifacts |

跑完看 Cell 9 的 `ablation_season_oof`（穩健指標）與
`ablation_holdout_vs_oof.png`（報告核心圖）。

In [ ]:
# === Cell 2 — 套件 + 路徑 + 內嵌球場 lookup（每個 runtime 必跑）===
import subprocess, sys, os
from pathlib import Path
subprocess.run([sys.executable,"-m","pip","install","-q",
                "lightgbm","shap","xgboost"], check=True)
import sklearn, xgboost, lightgbm, shap, pandas, numpy
print("sklearn",sklearn.__version__,"| xgb",xgboost.__version__,
      "| lgb",lightgbm.__version__,"| shap",shap.__version__,
      "| pandas",pandas.__version__)

ROOT = Path.cwd()                       # 所有 step 都以 cwd 為根
for d in ["data/raw/_lookup","data/processed","Results/eval",
          "Results/figures","models"]:
    (ROOT/d).mkdir(parents=True, exist_ok=True)

# step1b 需要球場->經緯度；不 clone repo，故內嵌寫出
LOOKUP_CSV = """stadium_norm,stadium_raw,station_id,station_name,latitude,longitude,note
樂天桃園,樂天桃園棒球場,C0C480,桃園,24.9429,121.2263,exposed; prevailing NW wind in cold months
洲際,臺中市洲際棒球場,72T250,臺中,24.1903,120.6791,bowl shape; moderate wind
天母,臺北市立天母棒球場,466920,臺北,25.1217,121.5324,humid; close to coast
新莊,新北市立新莊棒球場,C0AC60,新莊,25.0460,121.4521,northerly wind off coast
澄清湖,澄清湖棒球場,C0V250,鳳山,22.6738,120.3659,warm humid south
臺南,臺南市立棒球場,467410,臺南,22.9669,120.2138,humid; southerly winds
大巨蛋,臺北大巨蛋,466920,臺北,25.0396,121.5602,indoor; weather is climate context only (is_indoor=1)
其他_嘉義,嘉義市立棒球場,467480,嘉義,23.4833,120.4525,inland hot summer
其他_花蓮,花蓮縣立德興棒球場,466990,花蓮,23.9851,121.6092,coastal east
其他_臺東,臺東棒球村第一棒球場,467660,臺東,22.7596,121.1500,coastal south-east
其他_斗六,斗六棒球場,467480,嘉義,23.7106,120.5460,proxied via 嘉義 station (~25km west)
"""
(ROOT/"data/raw/_lookup/stadium_to_station.csv").write_text(
    LOOKUP_CSV, encoding="utf-8")
print("wrote stadium_to_station.csv (11 venues)")

In [ ]:
# === Cell 3 — 抓 rebas 資料（2024+2023；每個新 runtime 必跑）===
import os, subprocess, glob
USE_2023 = True   # 預設 True：單 2024 已實證無樣本外訊號；2023 把 N 366→~678

B = "https://github.com/rebas-tw/rebas.tw-open-data/releases/download"
urls = [
    f"{B}/v0.1.0-2024/CPBL-2024-OpenData.zip",
    f"{B}/v0.1.0-2024/CPBL-2024-Challenge-OpenData.zip",
    f"{B}/v0.1.0-2024/CPBL-2024-TaiwanSeries-OpenData.zip",
]
if USE_2023:
    urls += [
        f"{B}/v0.1.0-2023.0/CPBL-2023-G1-G150-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-G151-G300-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-Challenge-OpenData.zip",
        f"{B}/v0.1.0-2023.1/CPBL-2023-TaiwanSeries-OpenData.zip",
    ]
for u in urls:
    stem = u.split("/")[-1][:-4]
    fn = f"data/raw/{stem}.zip"
    subprocess.run(["wget","-q","-O",fn,u], check=True)
    subprocess.run(["unzip","-o","-q",fn,"-d",f"data/raw/{stem}"], check=True)
# 2024=ASCII (CPBL-2024-OpenData.json) / 2023=中文 (中職2023年-OpenData.json
# 等4種)；只配共同 token 'OpenData'，自動排除 per-game *-G<N>.json
js = sorted(glob.glob("data/raw/**/*OpenData*.json", recursive=True))
print(len(js), "combined JSON:")
for j in js: print("  ", j)
need = 7 if USE_2023 else 3
assert len(js) >= need, f"❌ 預期 >= {need} 個合併檔，只有 {len(js)} — 看上面清單"

In [ ]:
# === Cell 4 — 投手資料診斷（驗證 rebas pitcherBox schema）===
import json, glob, collections, statistics
files = sorted(glob.glob("data/raw/**/*OpenData*.json", recursive=True))
games = []
for f in files: games += json.load(open(f))
print("files:", [f.split("/")[-1] for f in files], "| total games:", len(games))
g = games[0]
print("game keys:", list(g.keys()))
pb = g.get("homePitcherBox") or []
print("homePitcherBox rows:", len(pb))
print("ONE pitcher row:", json.dumps(pb[0], ensure_ascii=False) if pb else "NONE")
bad=0; starts=collections.Counter(); team_sp=collections.defaultdict(set)
seasons=collections.Counter()
for gg in games:
    seasons[str(gg.get("seasonId") or gg.get("season"))[:4]] += 1
    for side,tk in (("homePitcherBox","homeTeam"),("awayPitcherBox","awayTeam")):
        rows=gg.get(side) or []
        s=[r for r in rows if r.get("order")==1]
        if len(s)!=1: bad+=1
        elif s:
            pid=s[0].get("playerId"); starts[pid]+=1; team_sp[gg.get(tk)].add(pid)
sv=list(starts.values())
print("games by season:", dict(seasons))
print("game-sides WITHOUT exactly one order==1:", bad, "/", 2*len(games))
print("distinct starters:", len(sv),
      "| starts/starter mean/median/max:",
      round(statistics.mean(sv),1) if sv else 0,
      statistics.median(sv) if sv else 0, max(sv) if sv else 0)
print("starters per team:", {t:len(s) for t,s in team_sp.items()})

In [ ]:
# === STEP 1 — build raw_games (inlined verbatim from scripts/step1_build_raw_games.py) ===
import json
import hashlib
import re
from datetime import datetime
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
RAW_ROOT = ROOT / "data/raw"
OUT = ROOT / "data/processed"
OUT.mkdir(parents=True, exist_ok=True)
PROV = ROOT / "data/raw/_provenance"
PROV.mkdir(parents=True, exist_ok=True)


def discover_sources(raw_root: Path):
    """Find every combined rebas OpenData JSON under data/raw/ and tag its
    game_type and season from the filename. Returns list[(game_type, season,
    path)] sorted by (season, type-rank) so the time-ordered build is stable.

    NOTE: the 2024 release uses ASCII names (CPBL-2024-OpenData.json); the
    2023 release uses Chinese names (中職2023年-OpenData.json /
    中職2023年下半季-OpenData.json / 中職2023年-季後挑戰賽-OpenData.json /
    中職2023年-台灣大賽-OpenData.json). Match on the shared 'OpenData' token
    only — this still excludes the per-game *-G<N>.json files (no 'OpenData'
    in their names). A 'CPBL-*' prefix would silently drop all 2023 data."""
    type_rank = {"regular": 0, "challenge": 1, "series": 2}
    found = []
    for p in sorted(raw_root.rglob("*OpenData*.json")):
        name = p.name
        if "Challenge" in name or "挑戰" in name:        # 季後挑戰賽
            gtype = "challenge"
        elif "TaiwanSeries" in name or "Series" in name or "台灣大賽" in name:
            gtype = "series"
        else:
            gtype = "regular"
        m = re.search(r"(20\d{2})", name)               # CPBL-2024- / 中職2023年-
        season = int(m.group(1)) if m else 0
        found.append((gtype, season, p))
    found.sort(key=lambda t: (t[1], type_rank.get(t[0], 9)))
    return found


SOURCES = [(g, p) for (g, _s, p) in discover_sources(RAW_ROOT)]
if not SOURCES:
    raise FileNotFoundError(
        f"No *OpenData*.json found under {RAW_ROOT}. "
        "Unzip rebas releases into data/raw/ first."
    )
print("discovered sources:")
for g, p in SOURCES:
    print(f"  [{g:9s}] {p.relative_to(ROOT)}")

# 11 stadiums in source -> 8 normalized levels.
# Bill-James park factor needs N>=20 ideally; the bottom 4 venues all have
# N<10 in 2024 -> collapse to "其他" to avoid overfit on tiny cells.
STADIUM_MAP = {
    "樂天桃園棒球場":     "樂天桃園",
    "臺中市洲際棒球場":   "洲際",
    "臺北市立天母棒球場": "天母",
    "新北市立新莊棒球場": "新莊",
    "澄清湖棒球場":       "澄清湖",
    "臺南市立棒球場":     "臺南",
    "臺北大巨蛋":         "大巨蛋",
    "嘉義市立棒球場":     "其他",
    "花蓮縣立德興棒球場": "其他",
    "臺東棒球村第一棒球場": "其他",
    "斗六棒球場":         "其他",
}
INDOOR_STADIUMS = {"大巨蛋"}

# batter-box stat columns we will re-aggregate per side
BATTER_STAT_KEYS = ["PA", "AB", "R", "H", "RBI", "2B", "3B", "HR",
                    "BB", "IBB", "HBP", "SO", "SH", "SF", "GIDP", "SB", "CS", "E"]

# pitcher-box stat columns (verified against real rebas 2023+2024 rows:
# Cell-4b diagnostic showed every game-side has exactly ONE order==1 row;
# fields IPOuts/NP/BF/H/HR/BB/IBB/HB/SO/R/ER). Staff = sum of all pitchers
# on a side; starter = the order==1 row. step2 turns these into leak-free
# PRIOR-game rolling form (we never use the current game's line for it).
PITCHER_STAT_KEYS = ["IPOuts", "NP", "BF", "H", "HR", "BB",
                     "IBB", "HB", "SO", "R", "ER"]
STARTER_STAT_KEYS = ["IPOuts", "ER", "H", "HR", "BB", "SO", "BF", "NP"]


def sum_inning_scores(arr):
    """Robust sum that skips non-numeric inning entries (e.g. 'X')."""
    s = 0
    for v in arr:
        try:
            s += int(v)
        except (ValueError, TypeError):
            continue
    return s


def aggregate_box(box, prefix):
    out = {f"{prefix}_{k}": 0 for k in BATTER_STAT_KEYS}
    for batter in box:
        for k in BATTER_STAT_KEYS:
            v = batter.get(k, 0) or 0
            try:
                out[f"{prefix}_{k}"] += int(v)
            except (ValueError, TypeError):
                pass
    return out


def aggregate_pitchers(box, prefix):
    """Team pitching-staff totals (sum of every pitcher that appeared).
    'p' prefix keeps these distinct from batter columns (home_pH = hits
    ALLOWED by home pitchers, vs home_H = hits BY home batters)."""
    out = {f"{prefix}_p{k}": 0 for k in PITCHER_STAT_KEYS}
    for pit in box:
        for k in PITCHER_STAT_KEYS:
            v = pit.get(k, 0) or 0
            try:
                out[f"{prefix}_p{k}"] += int(v)
            except (ValueError, TypeError):
                pass
    return out


def extract_starter(box, prefix):
    """The starting pitcher's own line for THIS game = the order==1 row
    (Cell-4b verified: exactly one per side, 0/1356 exceptions). Defensive
    min-by-order so a malformed box still yields the earliest appearance.
    sp_id (playerId) lets step2 roll each starter's OWN prior form."""
    out = {f"{prefix}_sp_id": None}
    out.update({f"{prefix}_sp_{k}": 0 for k in STARTER_STAT_KEYS})
    if not box:
        return out
    sp = min(box, key=lambda r: r.get("order") if r.get("order") is not None else 999)
    out[f"{prefix}_sp_id"] = sp.get("playerId")
    for k in STARTER_STAT_KEYS:
        v = sp.get(k, 0) or 0
        try:
            out[f"{prefix}_sp_{k}"] = int(v)
        except (ValueError, TypeError):
            pass
    return out


def make_game_id(game, game_type, seq_in_source):
    """Build a stable game_id: {date}-{type}-{seq}."""
    date = (game.get("date") or "")[:10].replace("-", "")
    return f"{date}-{game_type[:3].upper()}-{seq_in_source:03d}"


def parse_date(s):
    if not s:
        return pd.NaT
    s = s.replace("/", "-")
    # rebas pattern: "2024-04-04 17:05:00"
    m = re.match(r"^(\d{4})-(\d{1,2})-(\d{1,2})(?:\s+(\d{1,2}):(\d{2})(?::\d{2})?)?", s)
    if not m:
        return pd.NaT
    y, mo, d, hh, mm = m.groups()
    hh = hh or "00"; mm = mm or "00"
    return datetime(int(y), int(mo), int(d), int(hh), int(mm))


# ---------------------------------------------------------------------------
rows = []
provenance = {
    "run_id": datetime.utcnow().isoformat() + "Z",
    "rebas_release_tag": "v0.1.0-2024",
    "sources": {},
}

for game_type, path in SOURCES:
    with open(path, encoding="utf-8") as f:
        data = json.load(f)
    provenance["sources"][game_type] = {
        "path": str(path.relative_to(ROOT)),
        "n_games": len(data),
        "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
    }

    for seq, g in enumerate(data, start=1):
        home_scores = g.get("homeScores", [])
        away_scores = g.get("awayScores", [])
        h_total = sum_inning_scores(home_scores)
        a_total = sum_inning_scores(away_scores)

        home_box = g.get("homeBatterBox", []) or []
        away_box = g.get("awayBatterBox", []) or []
        home_pbox = g.get("homePitcherBox", []) or []
        away_pbox = g.get("awayPitcherBox", []) or []

        row = {
            "game_id":    make_game_id(g, game_type, seq),
            "game_type":  game_type,
            "season":     g.get("season"),
            "seasonId":   g.get("seasonId"),
            "seq":        g.get("seq"),
            "datetime":   g.get("date"),
            "stadium_raw": g.get("stadium"),
            "stadium":    STADIUM_MAP.get(g.get("stadium"), "其他"),
            "is_indoor":  int(STADIUM_MAP.get(g.get("stadium"), "其他") in INDOOR_STADIUMS),
            "home_team":  g.get("homeTeam"),
            "away_team":  g.get("awayTeam"),
            "home_team_id": g.get("homeTeamId"),
            "away_team_id": g.get("awayTeamId"),
            "home_innings_played": len(home_scores),
            "away_innings_played": len(away_scores),
            "home_score": h_total,
            "away_score": a_total,
            "total_score": h_total + a_total,
            "is_home_win": int(h_total > a_total),
            "is_tie":      int(h_total == a_total),
            "n_home_batters": len(home_box),
            "n_away_batters": len(away_box),
            "n_home_pitchers": len(home_pbox),
            "n_away_pitchers": len(away_pbox),
        }
        row.update(aggregate_box(home_box, "home"))
        row.update(aggregate_box(away_box, "away"))
        row.update(aggregate_pitchers(home_pbox, "home"))
        row.update(aggregate_pitchers(away_pbox, "away"))
        row.update(extract_starter(home_pbox, "home"))
        row.update(extract_starter(away_pbox, "away"))
        rows.append(row)

df = pd.DataFrame(rows)
df["date"] = df["datetime"].apply(parse_date)
df = df.sort_values(["date", "game_id"]).reset_index(drop=True)

# ---- sanity ----------------------------------------------------------------
print(f"total rows: {len(df)}")
print(f"by game_type: {df['game_type'].value_counts().to_dict()}")
print(f"ties dropped: {df['is_tie'].sum()}")

# 平手場直接丟掉, 我們是 binary classification
df = df.loc[df["is_tie"] == 0].drop(columns=["is_tie"]).reset_index(drop=True)
print(f"after dropping ties: {len(df)}")
print(f"home_win rate: {df['is_home_win'].mean():.3f}")
print(f"stadium normalize roster:")
print(df["stadium"].value_counts().to_string())

# ---- output ----------------------------------------------------------------
out_path = OUT / "raw_games.csv"
df.to_csv(out_path, index=False, encoding="utf-8")
print(f"\nwritten: {out_path}  shape={df.shape}")

provenance["output"] = {
    "path": str(out_path.relative_to(ROOT)),
    "rows": len(df),
    "cols": list(df.columns),
    "sha256": hashlib.sha256(out_path.read_bytes()).hexdigest(),
}
(PROV / "manifest_step1.json").write_text(
    json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8"
)
print(f"manifest: {PROV / 'manifest_step1.json'}")

In [ ]:
# === STEP 1b — Open-Meteo weather (inlined verbatim from scripts/step1b_fetch_weather.py) ===
import hashlib
import json
import time
import urllib.parse
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
RAW_GAMES = ROOT / "data/processed/raw_games.csv"
LOOKUP = ROOT / "data/raw/_lookup/stadium_to_station.csv"
OUT = ROOT / "data/processed/games_with_weather.csv"
CACHE = ROOT / "data/raw/.cache_weather"
CACHE.mkdir(parents=True, exist_ok=True)

API_URL = "https://archive-api.open-meteo.com/v1/archive"
HOURLY = "temperature_2m,relative_humidity_2m,wind_speed_10m,wind_direction_10m,precipitation"
RETRY_N = 3
DEFAULT_FIRST_PITCH_HOUR = 18   # used only when a game row has no clock time

WIND_BINS = [-1, 22.5, 67.5, 112.5, 157.5, 202.5, 247.5, 292.5, 337.5, 361]
WIND_LABELS = ["N", "NE", "E", "SE", "S", "SW", "W", "NW", "N"]


# ---------------------------------------------------------------------------
# Load games + stadium geo lookup
# ---------------------------------------------------------------------------
games = pd.read_csv(RAW_GAMES)
lookup = pd.read_csv(LOOKUP)

geo = lookup[["stadium_raw", "latitude", "longitude"]].drop_duplicates("stadium_raw")
games = games.merge(geo, on="stadium_raw", how="left")

n_missing_geo = int(games["latitude"].isna().sum())
if n_missing_geo:
    miss = sorted(games.loc[games["latitude"].isna(),
                            "stadium_raw"].dropna().unique())
    # HARD FAIL — never silently write NA weather (that exact "warn then
    # exit 0" pattern is what made the old R fetch_cwa.R waste a whole
    # Colab session). New seasons can introduce venues absent from the
    # lookup; force the user to fix the lookup before any downstream run.
    raise SystemExit(
        f"STOP: {n_missing_geo} games have no lat/lon. Unmapped "
        f"stadium_raw: {miss}. Add these rows (with latitude/longitude) "
        f"to data/raw/_lookup/stadium_to_station.csv and rerun step1b."
    )

# Game datetime: raw rebas string lives in `datetime`; floor to the hour.
# Some rows may carry only a date -> assume the default first-pitch hour.
dt = pd.to_datetime(games["datetime"], errors="coerce")
no_time = dt.dt.hour.eq(0) & dt.dt.minute.eq(0)
dt = dt.mask(no_time, dt.dt.normalize() + pd.Timedelta(hours=DEFAULT_FIRST_PITCH_HOUR))
games["game_dt"] = dt
games["join_hour"] = games["game_dt"].dt.floor("h").dt.strftime("%Y-%m-%dT%H:00")
games["game_date"] = games["game_dt"].dt.date

valid_dates = games["game_dt"].dropna()
date_lo = valid_dates.min().strftime("%Y-%m-%d")
date_hi = valid_dates.max().strftime("%Y-%m-%d")
print(f"games={len(games)}  missing geo={n_missing_geo}  "
      f"date range {date_lo} .. {date_hi}")


# ---------------------------------------------------------------------------
# Open-Meteo fetch with disk cache + retry
# ---------------------------------------------------------------------------
def fetch_openmeteo(lat, lon, start_date, end_date):
    key = hashlib.sha256(
        f"{lat:.4f}_{lon:.4f}_{start_date}_{end_date}".encode()
    ).hexdigest()[:16]
    cache_f = CACHE / f"{key}.json"
    if cache_f.exists():
        body = json.loads(cache_f.read_text())
    else:
        qs = urllib.parse.urlencode({
            "latitude": lat, "longitude": lon,
            "start_date": start_date, "end_date": end_date,
            "hourly": HOURLY, "timezone": "Asia/Taipei",
            "wind_speed_unit": "ms",
        })
        url = f"{API_URL}?{qs}"
        last_err = None
        body = None
        for attempt in range(RETRY_N):
            try:
                with urllib.request.urlopen(url, timeout=60) as resp:
                    if resp.status != 200:
                        raise RuntimeError(f"HTTP {resp.status}")
                    body = json.loads(resp.read().decode())
                cache_f.write_text(json.dumps(body))
                break
            except Exception as e:                       # noqa: BLE001
                last_err = e
                time.sleep(2 ** attempt)
        if body is None:
            raise RuntimeError(f"Open-Meteo failed after {RETRY_N} tries: {last_err}")

    h = body.get("hourly")
    if not h:
        return pd.DataFrame()
    return pd.DataFrame({
        "obs_dt": h["time"],
        "temperature": h["temperature_2m"],
        "humidity": h["relative_humidity_2m"],
        "wind_speed": h["wind_speed_10m"],
        "wind_dir": h["wind_direction_10m"],
        "precip": h["precipitation"],
    })


sites = games.dropna(subset=["latitude"]).drop_duplicates("stadium_raw")[
    ["stadium_raw", "latitude", "longitude"]
]
print(f"distinct sites to query: {len(sites)}")

weather_parts = []
for _, s in sites.iterrows():
    try:
        w = fetch_openmeteo(s["latitude"], s["longitude"], date_lo, date_hi)
    except Exception as e:                                # noqa: BLE001
        print(f"  fetch FAILED {s['stadium_raw']}: {e}")
        continue
    if len(w):
        w["stadium_raw"] = s["stadium_raw"]
        weather_parts.append(w)
        print(f"  {s['stadium_raw']:18s} rows={len(w)}")

# ---------------------------------------------------------------------------
# Join hourly obs to each game
# ---------------------------------------------------------------------------
WCOLS = ["temperature", "humidity", "wind_speed", "wind_dir", "precip"]
if weather_parts:
    weather = pd.concat(weather_parts, ignore_index=True)
    games_w = games.merge(
        weather.rename(columns={"obs_dt": "join_hour"}),
        on=["stadium_raw", "join_hour"], how="left",
    )
else:
    print("WARNING: Open-Meteo returned nothing. Writing NA weather columns.")
    games_w = games.copy()
    for c in WCOLS:
        games_w[c] = np.nan

games_w["wind_dir_cat"] = pd.cut(
    games_w["wind_dir"], bins=WIND_BINS, labels=WIND_LABELS, ordered=False
).astype("object")

miss_rate = float(games_w["temperature"].isna().mean())
print(f"\nweather join missing rate = {miss_rate:.4f}")
if miss_rate > 0.50:
    # geo matched but the hour-key join collapsed -> timezone / time-format
    # bug. Refuse to poison step2/step3 with all-NA weather.
    raise SystemExit(
        f"STOP: weather missing rate {miss_rate:.2%} > 50% — the "
        f"(stadium_raw, join_hour) join is broken (timezone or Open-Meteo "
        f"time-format mismatch), not just a few gaps. Not writing output."
    )
if miss_rate > 0.10:
    print("WARNING: >10% games lack weather — check timezone / coord lookup.")

drop_helpers = ["game_dt", "game_date"]
games_w = games_w.drop(columns=[c for c in drop_helpers if c in games_w.columns])
games_w.to_csv(OUT, index=False, encoding="utf-8")
print(f"written: {OUT}  shape={games_w.shape}")
print(games_w[["game_id", "stadium_raw", "join_hour"] + WCOLS].head(3).to_string(index=False))

In [ ]:
# === STEP 2 — feature engineering (inlined verbatim from scripts/step2_features.py) ===
import json
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
WEATHER_CSV = ROOT / "data/processed/games_with_weather.csv"
RAW_CSV = ROOT / "data/processed/raw_games.csv"
IN_CSV = WEATHER_CSV if WEATHER_CSV.exists() else RAW_CSV
OUT_CSV = ROOT / "data/processed/model_ready_data.csv"
PF_CSV = ROOT / "data/processed/park_factors.csv"

K_FACTOR = 4
HFA_ELO = 24
MOV_ALPHA = 0.6
MOV_BETA = 2.2
INIT_RATING = 1500.0
PYTHAG_WINDOW = 30
PYTHAG_EXPONENT = 1.83
REST_CAP = 5
ROLL_WINDOW = 30
MIN_PA_FOR_OPS = 50    # cutoff before OPS rolling becomes trustworthy
STAFF_PWINDOW = 30     # team pitching-staff rolling window (team-games)
STAFF_MIN_PRIOR = 5    # warm-up before staff rolling is trustworthy
SP_PWINDOW = 5         # per-starter rolling = that pitcher's last 5 starts
SP_MIN_PRIOR = 3       # < this many prior starts -> NaN (step3 median-imputes
                       #   = league fallback; median is only ~10 starts/pitcher)


# ============================================================================
# 1. Load & basic columns
# ============================================================================
df = pd.read_csv(IN_CSV, parse_dates=["date"])
df = df.sort_values(["date", "game_id"]).reset_index(drop=True)
WEATHER_FEATURES = [c for c in ["temperature", "humidity", "wind_speed",
                                "precip", "is_indoor"] if c in df.columns]
print(f"loaded {len(df)} games from {IN_CSV.name}")
print(f"weather features present: {WEATHER_FEATURES or 'NONE (run step1b)'}")


# ============================================================================
# 2. Elo (HFA + MoV adjusted, K=4)
# ============================================================================
def compute_elo(df_in):
    ratings = {}
    home_pre, away_pre = [], []
    home_post, away_post = [], []
    for h, a, hs, az in zip(df_in["home_team"], df_in["away_team"],
                            df_in["home_score"], df_in["away_score"]):
        h_pre = ratings.get(h, INIT_RATING)
        a_pre = ratings.get(a, INIT_RATING)
        home_pre.append(h_pre); away_pre.append(a_pre)
        expected_home = 1 / (1 + 10 ** ((a_pre - (h_pre + HFA_ELO)) / 400))
        if hs == az:
            home_post.append(h_pre); away_post.append(a_pre); continue
        actual_home = int(hs > az)
        diff_abs = abs(hs - az)
        rating_diff_winner = (h_pre + HFA_ELO - a_pre) if actual_home else (a_pre - h_pre - HFA_ELO)
        mov_mult = np.log(diff_abs + 1) * (MOV_BETA / (rating_diff_winner * MOV_ALPHA + MOV_BETA))
        mov_mult = max(0.5, min(4.0, mov_mult))
        delta = K_FACTOR * mov_mult * (actual_home - expected_home)
        h_post = h_pre + delta
        a_post = a_pre - delta
        home_post.append(h_post); away_post.append(a_post)
        ratings[h], ratings[a] = h_post, a_post
    df_in["home_elo_pre"] = home_pre
    df_in["away_elo_pre"] = away_pre
    df_in["home_elo_post"] = home_post
    df_in["away_elo_post"] = away_post
    return df_in


df = compute_elo(df)
df["diff_elo"] = df["home_elo_pre"] - df["away_elo_pre"]


# ============================================================================
# 3. Pythagenpat 30-game rolling — per-team perspective
# ============================================================================
def rolling_pythag(df_in, window=30, expo=1.83):
    long = []
    for i, r in df_in.iterrows():
        long.append({"game_id": r["game_id"], "date": r["date"], "side": "home",
                     "team": r["home_team"], "rs": r["home_score"], "ra": r["away_score"]})
        long.append({"game_id": r["game_id"], "date": r["date"], "side": "away",
                     "team": r["away_team"], "rs": r["away_score"], "ra": r["home_score"]})
    L = pd.DataFrame(long).sort_values(["team", "date", "game_id"]).reset_index(drop=True)
    pythag = []
    for team, g in L.groupby("team"):
        for i in range(len(g)):
            start = max(0, i - window)
            prev = g.iloc[start:i]
            if len(prev) < 5:
                pythag.append((g.iloc[i]["game_id"], g.iloc[i]["side"], np.nan))
            else:
                rs, ra = prev["rs"].sum(), prev["ra"].sum()
                if rs + ra == 0:
                    pythag.append((g.iloc[i]["game_id"], g.iloc[i]["side"], 0.5))
                else:
                    val = (rs ** expo) / (rs ** expo + ra ** expo)
                    pythag.append((g.iloc[i]["game_id"], g.iloc[i]["side"], val))
    P = pd.DataFrame(pythag, columns=["game_id", "side", "pythag_pre"])
    wide = P.pivot(index="game_id", columns="side", values="pythag_pre").reset_index()
    wide = wide.rename(columns={"home": "home_pythag_30g", "away": "away_pythag_30g"})
    return df_in.merge(wide, on="game_id", how="left")


df = rolling_pythag(df, window=PYTHAG_WINDOW, expo=PYTHAG_EXPONENT)
df["diff_pythag"] = df["home_pythag_30g"] - df["away_pythag_30g"]


# ============================================================================
# 4. rest_days  (cap 5)
# ============================================================================
def rest_days(df_in, cap=5):
    long = []
    for _, r in df_in.iterrows():
        long.append({"game_id": r["game_id"], "date": r["date"], "side": "home", "team": r["home_team"]})
        long.append({"game_id": r["game_id"], "date": r["date"], "side": "away", "team": r["away_team"]})
    L = pd.DataFrame(long).sort_values(["team", "date", "game_id"]).reset_index(drop=True)
    L["prev_date"] = L.groupby("team")["date"].shift(1)
    L["rest_days"] = (L["date"] - L["prev_date"]).dt.days.clip(upper=cap)
    wide = L.pivot(index="game_id", columns="side", values="rest_days").reset_index()
    wide = wide.rename(columns={"home": "home_rest_days", "away": "away_rest_days"})
    return df_in.merge(wide, on="game_id", how="left")


df = rest_days(df, cap=REST_CAP)
df["diff_rest"] = df["home_rest_days"] - df["away_rest_days"]


# ============================================================================
# 5. Team-game rolling 30-game batter-state features
# ============================================================================
def rolling_team_batter_state(df_in, window=ROLL_WINDOW):
    # long-form per game per side
    rows = []
    for _, r in df_in.iterrows():
        for side in ("home", "away"):
            other = "away" if side == "home" else "home"
            rs = r[f"{side}_score"]
            ra = r[f"{other}_score"]
            rows.append({
                "game_id": r["game_id"], "date": r["date"], "side": side,
                "team": r[f"{side}_team"], "stadium": r["stadium"],
                "PA": r[f"{side}_PA"], "AB": r[f"{side}_AB"],
                "H":  r[f"{side}_H"],  "HR": r[f"{side}_HR"],
                "2B": r[f"{side}_2B"], "3B": r[f"{side}_3B"],
                "BB": r[f"{side}_BB"], "HBP": r[f"{side}_HBP"],
                "SF": r[f"{side}_SF"], "SO": r[f"{side}_SO"],
                "rs": rs, "ra": ra,
            })
    L = pd.DataFrame(rows).sort_values(["team", "date", "game_id"]).reset_index(drop=True)

    stat_cols = ["PA", "AB", "H", "HR", "2B", "3B", "BB", "HBP", "SF", "SO", "rs"]
    feat_rows = []
    for team, g in L.groupby("team"):
        g = g.reset_index(drop=True)
        for i in range(len(g)):
            start = max(0, i - window)
            prev = g.iloc[start:i]
            row = {"game_id": g.iloc[i]["game_id"], "side": g.iloc[i]["side"]}
            if len(prev) < 5:
                for c in ["AVG", "OBP", "SLG", "OPS", "ISO", "K_pct", "BB_pct",
                          "HR_per_g", "runs_per_g"]:
                    row[c] = np.nan
            else:
                S = prev[stat_cols].sum()
                S_1B = S["H"] - S["2B"] - S["3B"] - S["HR"]
                ab_den = S["AB"]
                pa_den = S["PA"]
                obp_den = S["AB"] + S["BB"] + S["HBP"] + S["SF"]
                avg = S["H"] / ab_den if ab_den else np.nan
                obp = (S["H"] + S["BB"] + S["HBP"]) / obp_den if obp_den else np.nan
                slg = (S_1B + 2 * S["2B"] + 3 * S["3B"] + 4 * S["HR"]) / ab_den if ab_den else np.nan
                row["AVG"] = avg
                row["OBP"] = obp
                row["SLG"] = slg
                row["OPS"] = (obp + slg) if (obp is not None and slg is not None) else np.nan
                row["ISO"] = (slg - avg) if (slg is not None and avg is not None) else np.nan
                row["K_pct"] = S["SO"] / pa_den if pa_den else np.nan
                row["BB_pct"] = S["BB"] / pa_den if pa_den else np.nan
                row["HR_per_g"] = S["HR"] / len(prev)
                row["runs_per_g"] = S["rs"] / len(prev)
            feat_rows.append(row)
    F = pd.DataFrame(feat_rows)
    # pivot wide
    out = df_in.copy()
    for col in ["AVG", "OBP", "SLG", "OPS", "ISO", "K_pct", "BB_pct", "HR_per_g", "runs_per_g"]:
        h = F[F["side"] == "home"][["game_id", col]].rename(columns={col: f"home_{col}_30g"})
        a = F[F["side"] == "away"][["game_id", col]].rename(columns={col: f"away_{col}_30g"})
        out = out.merge(h, on="game_id", how="left").merge(a, on="game_id", how="left")
        out[f"diff_{col}_30g"] = out[f"home_{col}_30g"] - out[f"away_{col}_30g"]
    return out


df = rolling_team_batter_state(df, window=ROLL_WINDOW)


# ============================================================================
# 6. team_at_stadium_OPS_30g — per (team, stadium) rolling
# ============================================================================
def team_at_stadium_rolling(df_in, window=ROLL_WINDOW):
    rows = []
    for _, r in df_in.iterrows():
        for side in ("home", "away"):
            other = "away" if side == "home" else "home"
            rows.append({
                "game_id": r["game_id"], "date": r["date"], "side": side,
                "team": r[f"{side}_team"], "stadium": r["stadium"],
                "PA": r[f"{side}_PA"], "AB": r[f"{side}_AB"],
                "H":  r[f"{side}_H"],  "HR": r[f"{side}_HR"],
                "2B": r[f"{side}_2B"], "3B": r[f"{side}_3B"],
                "BB": r[f"{side}_BB"], "HBP": r[f"{side}_HBP"],
                "SF": r[f"{side}_SF"],
            })
    L = pd.DataFrame(rows).sort_values(["team", "stadium", "date", "game_id"]).reset_index(drop=True)
    out_rows = []
    for (team, stad), g in L.groupby(["team", "stadium"]):
        g = g.reset_index(drop=True)
        for i in range(len(g)):
            prev = g.iloc[max(0, i - window):i]
            if len(prev) < 3:
                ops = np.nan
            else:
                S = prev[["AB", "H", "2B", "3B", "HR", "BB", "HBP", "SF"]].sum()
                S_1B = S["H"] - S["2B"] - S["3B"] - S["HR"]
                ab_den = S["AB"]
                obp_den = S["AB"] + S["BB"] + S["HBP"] + S["SF"]
                if not ab_den or not obp_den:
                    ops = np.nan
                else:
                    obp = (S["H"] + S["BB"] + S["HBP"]) / obp_den
                    slg = (S_1B + 2 * S["2B"] + 3 * S["3B"] + 4 * S["HR"]) / ab_den
                    ops = obp + slg
            out_rows.append({"game_id": g.iloc[i]["game_id"], "side": g.iloc[i]["side"], "ops_at_stad": ops})
    F = pd.DataFrame(out_rows)
    h = F[F["side"] == "home"][["game_id", "ops_at_stad"]].rename(
        columns={"ops_at_stad": "home_at_stadium_OPS_30g"})
    a = F[F["side"] == "away"][["game_id", "ops_at_stad"]].rename(
        columns={"ops_at_stad": "away_at_stadium_OPS_30g"})
    return df_in.merge(h, on="game_id", how="left").merge(a, on="game_id", how="left")


df = team_at_stadium_rolling(df, window=ROLL_WINDOW)
df["diff_at_stadium_OPS"] = df["home_at_stadium_OPS_30g"] - df["away_at_stadium_OPS_30g"]


# ============================================================================
# 6b. Pitching rolling — team staff (30 team-games) + each starting
#     pitcher's OWN recent form (last 5 starts). Strictly prior games only
#     (leak-free): the starter's identity is known at first pitch — not
#     leakage — but only their PRIOR lines feed the feature, never this game.
#     ERA/WHIP/HR9 use IP = IPOuts/3; K%/BB% use batters-faced.
# ============================================================================
def _pitch_rates(prev, min_prior, pfx):
    keys = ["ERA", "WHIP", "K_pct", "BB_pct", "HR9"]
    if pfx == "sp":
        keys = keys + ["IPouts"]
    if len(prev) < min_prior:
        return {f"{pfx}{k}": np.nan for k in keys}
    S = prev[["IPOuts", "ER", "H", "HR", "BB", "SO", "BF"]].sum()
    ip = S["IPOuts"] / 3.0
    bf = S["BF"]
    out = {
        f"{pfx}ERA":    (9.0 * S["ER"] / ip) if ip > 0 else np.nan,
        f"{pfx}WHIP":   ((S["BB"] + S["H"]) / ip) if ip > 0 else np.nan,
        f"{pfx}K_pct":  (S["SO"] / bf) if bf > 0 else np.nan,
        f"{pfx}BB_pct": (S["BB"] / bf) if bf > 0 else np.nan,
        f"{pfx}HR9":    (9.0 * S["HR"] / ip) if ip > 0 else np.nan,
    }
    if pfx == "sp":
        out["spIPouts"] = prev["IPOuts"].mean()      # durability: outs/start
    return out


def _merge_pitch(df_in, F, metrics, suf):
    out = df_in
    for col in metrics:
        h = F[F["side"] == "home"][["game_id", col]].rename(
            columns={col: f"home_{col}{suf}"})
        a = F[F["side"] == "away"][["game_id", col]].rename(
            columns={col: f"away_{col}{suf}"})
        out = out.merge(h, on="game_id", how="left").merge(a, on="game_id", how="left")
        out[f"diff_{col}{suf}"] = out[f"home_{col}{suf}"] - out[f"away_{col}{suf}"]
    return out


def rolling_pitching(df_in, staff_window=STAFF_PWINDOW, sp_window=SP_PWINDOW,
                     sp_min_prior=SP_MIN_PRIOR, staff_min_prior=STAFF_MIN_PRIOR):
    # ---- A. team pitching-staff rolling, keyed by team (mirror batter-state).
    #         *_30g-suffixed -> shares the warm-up filter with batter-state.
    rows = []
    for _, r in df_in.iterrows():
        for side in ("home", "away"):
            rows.append({
                "game_id": r["game_id"], "date": r["date"], "side": side,
                "team": r[f"{side}_team"],
                "IPOuts": r[f"{side}_pIPOuts"], "ER": r[f"{side}_pER"],
                "H": r[f"{side}_pH"], "HR": r[f"{side}_pHR"],
                "BB": r[f"{side}_pBB"], "SO": r[f"{side}_pSO"],
                "BF": r[f"{side}_pBF"],
            })
    L = pd.DataFrame(rows).sort_values(["team", "date", "game_id"]).reset_index(drop=True)
    feat = []
    for _team, g in L.groupby("team"):
        g = g.reset_index(drop=True)
        for i in range(len(g)):
            prev = g.iloc[max(0, i - staff_window):i]
            row = {"game_id": g.iloc[i]["game_id"], "side": g.iloc[i]["side"]}
            row.update(_pitch_rates(prev, staff_min_prior, "staff"))
            feat.append(row)
    df_in = _merge_pitch(df_in, pd.DataFrame(feat),
                         ["staffERA", "staffWHIP", "staffK_pct",
                          "staffBB_pct", "staffHR9"], "_30g")

    # ---- B. per-starter rolling, keyed by sp_id (pool the pitcher's starts
    #         home OR away — pitching skill is venue-independent). Cold start
    #         (< sp_min_prior prior starts) -> NaN, deliberately left for
    #         step3's median imputer = the league-average fallback tier.
    #         _l5-suffixed (NOT _30g) so the warm-up filter does NOT drop
    #         every rookie / spot-start game (median ~10 starts/pitcher).
    rows = []
    for _, r in df_in.iterrows():
        for side in ("home", "away"):
            spid = r.get(f"{side}_sp_id")
            if spid is None or (isinstance(spid, float) and pd.isna(spid)):
                continue
            rows.append({
                "game_id": r["game_id"], "date": r["date"], "side": side,
                "sp_id": spid,
                "IPOuts": r[f"{side}_sp_IPOuts"], "ER": r[f"{side}_sp_ER"],
                "H": r[f"{side}_sp_H"], "HR": r[f"{side}_sp_HR"],
                "BB": r[f"{side}_sp_BB"], "SO": r[f"{side}_sp_SO"],
                "BF": r[f"{side}_sp_BF"],
            })
    L = pd.DataFrame(rows).sort_values(["sp_id", "date", "game_id"]).reset_index(drop=True)
    feat = []
    for _spid, g in L.groupby("sp_id"):
        g = g.reset_index(drop=True)
        for i in range(len(g)):
            prev = g.iloc[max(0, i - sp_window):i]
            row = {"game_id": g.iloc[i]["game_id"], "side": g.iloc[i]["side"]}
            row.update(_pitch_rates(prev, sp_min_prior, "sp"))
            feat.append(row)
    df_in = _merge_pitch(df_in, pd.DataFrame(feat),
                         ["spERA", "spWHIP", "spK_pct", "spBB_pct",
                          "spHR9", "spIPouts"], "_l5")
    return df_in


df = rolling_pitching(df)


# ============================================================================
# 7. Park Factor — time-aware, leave-one-out
# ============================================================================
def park_factor_time_aware(df_in):
    """For each game g at stadium s and date t:
       pf_pre = avg_total_score_at_s_before_t / avg_total_score_at_other_stadiums_before_t
       Falls back to 1.0 if < 5 prior games at this stadium.
    """
    df_in = df_in.copy()
    df_in["pf_pre"] = np.nan
    for i, r in df_in.iterrows():
        prev = df_in.iloc[:i]   # all games strictly before this row (date-sorted)
        same_stad = prev[prev["stadium"] == r["stadium"]]
        other_stad = prev[prev["stadium"] != r["stadium"]]
        if len(same_stad) < 5 or len(other_stad) < 5:
            df_in.at[i, "pf_pre"] = 1.0
        else:
            df_in.at[i, "pf_pre"] = (same_stad["total_score"].mean() /
                                      other_stad["total_score"].mean())
    return df_in


df = park_factor_time_aware(df)

# also produce a stadium summary table
pf_summary = (df.groupby("stadium")
                .agg(n=("game_id", "size"),
                     home_win_rate=("is_home_win", "mean"),
                     avg_total_score=("total_score", "mean"),
                     pf_end_of_season=("pf_pre", "last"))
                .round(3).sort_values("avg_total_score", ascending=False))
pf_summary.to_csv(PF_CSV)
print(f"\npark_factors:\n{pf_summary}")


# ============================================================================
# 8. Day-of-week / month niceties
# ============================================================================
df["dow"] = df["date"].dt.day_name()
df["month"] = df["date"].dt.month
df["is_weekend"] = df["date"].dt.day_name().isin(["Saturday", "Sunday"]).astype(int)


# ============================================================================
# 9. Final hygiene + output
# ============================================================================
# fill rest_days NA (first game of season for each team) with median
for col in ["home_rest_days", "away_rest_days"]:
    df[col] = df[col].fillna(df[col].median())
df["diff_rest"] = df["home_rest_days"] - df["away_rest_days"]

print(f"\nfinal shape: {df.shape}")
print(f"home_win rate: {df['is_home_win'].mean():.3f}")

# How many rows have all rolling features available (after warm-up)?
roll_cols = [c for c in df.columns if c.endswith("_30g")]
df["features_complete"] = (df[roll_cols].isna().sum(axis=1) == 0).astype(int)
print(f"features_complete (no NA in 30g rolling): {df['features_complete'].sum()} / {len(df)}")

df.to_csv(OUT_CSV, index=False, encoding="utf-8")
print(f"\nwritten: {OUT_CSV}")
print("columns added in step 2:")
new_cols = [c for c in df.columns if c.endswith(("_pre", "_30g", "_l5", "_pythag", "_rest_days", "_rest", "_elo", "_OPS")) or c in ("dow", "month", "is_weekend", "pf_pre", "diff_elo", "diff_pythag", "features_complete")]
for c in sorted(new_cols):
    print(f"  {c}")

In [ ]:
# === STEP 3 — m1–m7 + season-OOF + artifacts (inlined verbatim from scripts/step3_models.py) ===
import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score,
                             brier_score_loss, f1_score, log_loss,
                             roc_auc_score)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*ConvergenceWarning.*")

ROOT = Path.cwd()
IN_CSV = ROOT / "data/processed/model_ready_data.csv"
FIG = ROOT / "Results/figures"
EVAL = ROOT / "Results/eval"
MODELS = ROOT / "models"
for d in (FIG, EVAL, MODELS):
    d.mkdir(parents=True, exist_ok=True)

RNG = 42
TARGET = "is_home_win"

# ============================================================================
# Load + warm-up filter + time-aware split
# ============================================================================
df = pd.read_csv(IN_CSV, parse_dates=["date"])
print(f"loaded {len(df)} games  cols={df.shape[1]}")
df = df[df["features_complete"] == 1].reset_index(drop=True)
print(f"after warm-up filter: {len(df)} games  "
      f"home_win base rate={df[TARGET].mean():.3f}")

train = df[df["date"] < "2024-08-01"].reset_index(drop=True)
valid = df[(df["date"] >= "2024-08-01") & (df["date"] < "2024-09-16")].reset_index(drop=True)
test = df[df["date"] >= "2024-09-16"].reset_index(drop=True)
trainval = pd.concat([train, valid], ignore_index=True)
print(f"split: train={len(train)} valid={len(valid)} test={len(test)}")
print(f"home_win  train={train[TARGET].mean():.3f}  "
      f"valid={valid[TARGET].mean():.3f}  test={test[TARGET].mean():.3f}")

# ============================================================================
# Feature group catalogue
# ============================================================================
STADIUM_CAT = ["stadium"]
STADIUM_NUM = [c for c in ["is_indoor"] if c in df.columns]
WEATHER_COLS = [c for c in ["temperature", "humidity", "wind_speed", "precip"]
                if c in df.columns]
TEAM_STRENGTH = [c for c in ["diff_elo", "diff_pythag", "diff_rest", "pf_pre"]
                 if c in df.columns]
BATTER_STATE = [c for c in ["diff_OPS_30g", "diff_HR_per_g_30g",
                            "diff_K_pct_30g", "diff_BB_pct_30g",
                            "diff_runs_per_g_30g", "diff_at_stadium_OPS"]
                if c in df.columns]
# starter own last-5 form (sp*_l5, NaN-tolerant -> median-imputed) +
# team pitching-staff rolling 30g (staff*_30g, shares warm-up filter)
PITCHING = [c for c in ["diff_spERA_l5", "diff_spWHIP_l5",
                        "diff_spK_pct_l5", "diff_spBB_pct_l5",
                        "diff_spHR9_l5", "diff_spIPouts_l5",
                        "diff_staffERA_30g", "diff_staffWHIP_30g",
                        "diff_staffK_pct_30g", "diff_staffBB_pct_30g",
                        "diff_staffHR9_30g"]
            if c in df.columns]
STADIUM_ALL = STADIUM_CAT + STADIUM_NUM

if not WEATHER_COLS:
    print("NOTE: no weather columns -> run scripts/step1b_fetch_weather.py "
          "then step2 so m3/m6/m7 become meaningful. Continuing degraded.")

FEATURE_GROUPS = {
    "m1": [],
    "m2": STADIUM_ALL,
    "m3": WEATHER_COLS,
    "m4": TEAM_STRENGTH,
    "m5": BATTER_STATE,
    "m6": PITCHING,
    "m7": STADIUM_ALL + WEATHER_COLS + TEAM_STRENGTH + BATTER_STATE + PITCHING,
}


def build_preprocessor(feats):
    cat = [f for f in feats if f in STADIUM_CAT]
    num = [f for f in feats if f not in cat]
    transformers = []
    if num:
        transformers.append(("num", Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("sc", StandardScaler())]), num))
    if cat:
        transformers.append(("cat", Pipeline([
            ("imp", SimpleImputer(strategy="most_frequent")),
            ("oh", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))]), cat))
    return ColumnTransformer(transformers, remainder="drop")


def evaluate(y_true, p_hat, thr=0.5):
    y_true = np.asarray(y_true)
    y_pred = (p_hat >= thr).astype(int)
    return {
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "bal_acc": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": roc_auc_score(y_true, p_hat) if len(np.unique(y_true)) > 1 else float("nan"),
        "brier": brier_score_loss(y_true, p_hat),
        "log_loss": log_loss(y_true, p_hat, labels=[0, 1]),
    }


def auc_bootstrap_ci(y_true, p_hat, n_boot=1000, seed=RNG):
    y_true = np.asarray(y_true)
    p_hat = np.asarray(p_hat)
    rs = np.random.RandomState(seed)
    n = len(y_true)
    vals = []
    for _ in range(n_boot):
        idx = rs.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        vals.append(roc_auc_score(y_true[idx], p_hat[idx]))
    if not vals:
        return (float("nan"), float("nan"))
    return (float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5)))


def ts_oof_proba(estimator, X, y, splitter):
    """Walk-forward out-of-fold P(y=1). TimeSeriesSplit is NOT a partition
    (the first training block is never a test fold), so cross_val_predict
    rejects it with 'only works for partitions'. Do it by hand: clone + fit
    on each fold's past, predict its future. Rows never in any test fold
    stay NaN; the caller masks them."""
    X = X.reset_index(drop=True)
    y = pd.Series(np.asarray(y))
    oof = np.full(len(y), np.nan)
    for tr, te in splitter.split(X):
        est = clone(estimator)
        est.fit(X.iloc[tr], y.iloc[tr])
        oof[te] = est.predict_proba(X.iloc[te])[:, 1]
    return oof


# ============================================================================
# 1. Ablation m1..m7 (algorithm fixed = logistic regression)
# ============================================================================
print("\n" + "=" * 70 + "\nABLATION m1..m7 (logistic; features vary)\n" + "=" * 70)
ablation_rows = []
ablation_oof = {}                       # leak-free walk-forward per group
ablation_oof_folds = {}                 # per-fold AUC (sign-flip vs noise)
_ab_tscv = TimeSeriesSplit(n_splits=5)
for mname, feats in FEATURE_GROUPS.items():
    feats = [f for f in feats if f in df.columns]
    for split_name, sdf in [("train", train), ("valid", valid), ("test", test)]:
        y = sdf[TARGET]
        if mname == "m1" or not feats:
            p = np.full(len(y), trainval[TARGET].mean())
            note = "intercept" if mname == "m1" else "no-features-fallback"
        else:
            pipe = Pipeline([("pre", build_preprocessor(feats)),
                             ("clf", LogisticRegression(max_iter=2000, C=1.0,
                                                        solver="liblinear"))])
            pipe.fit(trainval[feats], trainval[TARGET])
            p = pipe.predict_proba(sdf[feats])[:, 1]
            note = f"{len(feats)}feat"
        m = evaluate(y, p)
        m.update({"model": mname, "split": split_name, "note": note})
        ablation_rows.append(m)
    # Per-group season-OOF: the ROBUST metric. The N=47 holdout AUC above
    # has a CI ~[.45,.81] — useless for ranking groups. This refits the
    # same logistic walk-forward over the whole post-warmup season so
    # "does pitching (m6) actually beat the HFA baseline out-of-sample?"
    # is answered by signal, not by a 47-game coin-flip. Cheap (logistic).
    if mname == "m1" or not feats:
        ablation_oof[mname] = float("nan")          # constant prior == chance
    else:
        oof_pipe = Pipeline([("pre", build_preprocessor(feats)),
                             ("clf", LogisticRegression(max_iter=2000, C=1.0,
                                                        solver="liblinear"))])
        o = ts_oof_proba(oof_pipe, df[feats], df[TARGET], _ab_tscv)
        k = ~np.isnan(o)
        ablation_oof[mname] = (
            roc_auc_score(df.loc[k, TARGET], o[k])
            if df.loc[k, TARGET].nunique() > 1 else float("nan"))
        # Fold-by-fold AUC: distinguishes "noise scattered around .50"
        # from a systematic <.50 sign-flip. Same verdict either way (no
        # robust signal) but the report must word it to match reality.
        yv = df[TARGET].values
        folds = []
        for _tr, _te in _ab_tscv.split(df):
            yt = yv[_te]
            folds.append(round(float(roc_auc_score(yt, o[_te])), 3)
                         if len(np.unique(yt)) > 1 else float("nan"))
        ablation_oof_folds[mname] = folds
ablation = pd.DataFrame(ablation_rows)
ablation["season_oof_auc"] = ablation["model"].map(ablation_oof)
ablation.to_csv(EVAL / "results_ablation.csv", index=False)
_ab_test = (ablation[ablation.split == "test"]
            .assign(season_oof=lambda d: d["model"].map(ablation_oof))
            .sort_values("season_oof_auc", ascending=False)
            [["model", "note", "n", "auc", "season_oof_auc", "brier"]])
print("(ranked by season_oof_auc — the robust metric; 'auc' is the "
      "noisy N=47 holdout)")
print(_ab_test.to_string(index=False))
print(f"\nm1 HFA-baseline season-OOF = chance (~0.50); "
      f"m6 pitching season-OOF = {ablation_oof.get('m6', float('nan')):.3f} ; "
      f"m7 full = {ablation_oof.get('m7', float('nan')):.3f}")
print(f"m6 per-fold OOF AUC = {ablation_oof_folds.get('m6')}  "
      "(scattered ~.50 => noise; systematically <.50 => small-N "
      "sign-flip — same verdict: no robust signal)")

# ============================================================================
# 2. Algorithm comparison @ full m7 (features fixed; algorithm varies)
# ============================================================================
print("\n" + "=" * 70 + "\nALGORITHMS @ m7 full features\n" + "=" * 70)
m7 = [f for f in FEATURE_GROUPS["m7"] if f in df.columns]
Xtv, ytv = trainval[m7], trainval[TARGET]
Xte, yte = test[m7], test[TARGET]

algos = {
    "logit": LogisticRegression(max_iter=2000, C=1.0, solver="liblinear"),
    "glmnet_l2": LogisticRegression(penalty="l2", C=0.3, max_iter=2000,
                                    solver="liblinear"),
    "glmnet_elastic": LogisticRegression(penalty="elasticnet", C=0.5,
                                          l1_ratio=0.5, max_iter=3000,
                                          solver="saga"),
    "rf": RandomForestClassifier(n_estimators=400, max_depth=6,
                                 min_samples_leaf=5, random_state=RNG,
                                 n_jobs=-1),
    "xgb": xgb.XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05,
                             subsample=0.8, colsample_bytree=0.8,
                             reg_alpha=0.1, reg_lambda=1.0,
                             eval_metric="logloss", random_state=RNG,
                             n_jobs=-1, tree_method="hist"),
    "lgb": lgb.LGBMClassifier(n_estimators=400, num_leaves=15,
                              learning_rate=0.05, subsample=0.8,
                              colsample_bytree=0.8, min_child_samples=10,
                              reg_alpha=0.1, reg_lambda=1.0,
                              random_state=RNG, n_jobs=-1, verbosity=-1),
}
algo_rows = []
for name, clf in algos.items():
    pipe = Pipeline([("pre", build_preprocessor(m7)), ("clf", clf)])
    pipe.fit(Xtv, ytv)
    m = evaluate(yte, pipe.predict_proba(Xte)[:, 1])
    m["model"] = name
    algo_rows.append(m)
    print(f"  {name:15s} auc={m['auc']:.3f} acc={m['accuracy']:.3f} "
          f"brier={m['brier']:.3f} ll={m['log_loss']:.3f}")
algo_df = pd.DataFrame(algo_rows).sort_values("auc", ascending=False)
algo_df.to_csv(EVAL / "results_algos.csv", index=False)

# ============================================================================
# 3. Tune candidates with TimeSeriesSplit; WINNER = best CV-AUC
# ============================================================================
print("\n" + "=" * 70 + "\nTUNING (TimeSeriesSplit n=5; winner by CV-AUC)\n" + "=" * 70)
tscv = TimeSeriesSplit(n_splits=5)
# n_jobs=1 INSIDE each estimator, n_jobs=4 on the search -> avoids the
# 4x4=16-thread thrash that stalled an earlier run on a 4-core box.
grids = {
    "xgb": (Pipeline([("pre", build_preprocessor(m7)),
                      ("clf", xgb.XGBClassifier(eval_metric="logloss",
                                                random_state=RNG, n_jobs=1,
                                                tree_method="hist",
                                                reg_alpha=0.1, reg_lambda=1.0,
                                                subsample=0.8,
                                                colsample_bytree=0.8))]),
            {"clf__n_estimators": [200, 400],
             "clf__max_depth": [2, 3, 4],
             "clf__learning_rate": [0.03, 0.05, 0.1]}),
    "lgb": (Pipeline([("pre", build_preprocessor(m7)),
                      ("clf", lgb.LGBMClassifier(subsample=0.8,
                                                 colsample_bytree=0.8,
                                                 min_child_samples=10,
                                                 reg_alpha=0.1, reg_lambda=1.0,
                                                 random_state=RNG, n_jobs=1,
                                                 verbosity=-1))]),
            {"clf__n_estimators": [200, 400],
             "clf__num_leaves": [8, 15, 31],
             "clf__learning_rate": [0.03, 0.05, 0.1]}),
    "elastic": (Pipeline([("pre", build_preprocessor(m7)),
                          ("clf", LogisticRegression(penalty="elasticnet",
                                                     solver="saga",
                                                     max_iter=5000,
                                                     random_state=RNG))]),
                {"clf__C": [0.1, 0.3, 1.0, 3.0],
                 "clf__l1_ratio": [0.2, 0.5, 0.8]}),
    "rf": (Pipeline([("pre", build_preprocessor(m7)),
                     ("clf", RandomForestClassifier(random_state=RNG,
                                                    n_jobs=1))]),
           {"clf__n_estimators": [200, 400, 800],
            "clf__max_depth": [4, 6, 8, None],
            "clf__min_samples_leaf": [3, 5, 10]}),
}
searches, cv_auc = {}, {}
for name, (pipe, grid) in grids.items():
    gs = GridSearchCV(pipe, grid, cv=tscv, scoring="roc_auc", n_jobs=4)
    gs.fit(Xtv, ytv)
    searches[name] = gs
    cv_auc[name] = gs.best_score_
    print(f"  {name:8s} CV-AUC={gs.best_score_:.3f}  {gs.best_params_}")

winner_name = max(cv_auc, key=cv_auc.get)
winner_search = searches[winner_name]
winner_fitted = winner_search.best_estimator_          # fitted on trainval
print(f"\nWINNER (by CV-AUC): {winner_name}  CV-AUC={cv_auc[winner_name]:.3f}")

# ============================================================================
# 4. Honest holdout — every tuned candidate + bootstrap CI
# ============================================================================
print("\n" + "=" * 70 + "\nHOLDOUT (test) — point estimate + 95% bootstrap CI\n" + "=" * 70)
final_rows, preds_for_calib = [], {}
for name, gs in searches.items():
    p = gs.best_estimator_.predict_proba(Xte)[:, 1]
    m = evaluate(yte, p)
    lo, hi = auc_bootstrap_ci(yte, p)
    m.update({"model": f"tuned_{name}", "cv_auc": cv_auc[name],
              "auc_ci_lo": lo, "auc_ci_hi": hi})
    final_rows.append(m)
    preds_for_calib[f"tuned_{name}"] = p
    print(f"  tuned_{name:8s} holdout-AUC={m['auc']:.3f} "
          f"[{lo:.3f}, {hi:.3f}]  CV-AUC={cv_auc[name]:.3f}")
final_df = pd.DataFrame(final_rows).sort_values("cv_auc", ascending=False)
final_df.to_csv(EVAL / "results_tuned.csv", index=False)

# ============================================================================
# 5. ONE calibration: isotonic via time-aware CV on train+valid
# ============================================================================
print("\n" + "=" * 70 + "\nCALIBRATION (isotonic, TimeSeriesSplit on train+valid)\n" + "=" * 70)
cal = CalibratedClassifierCV(clone(winner_fitted), method="isotonic",
                             cv=TimeSeriesSplit(n_splits=3))
cal.fit(Xtv, ytv)
p_raw = winner_fitted.predict_proba(Xte)[:, 1]
p_cal = cal.predict_proba(Xte)[:, 1]
m_raw = evaluate(yte, p_raw)
m_cal = evaluate(yte, p_cal)
preds_for_calib["winner_calibrated"] = p_cal
print(f"  raw        auc={m_raw['auc']:.3f} brier={m_raw['brier']:.3f} ll={m_raw['log_loss']:.3f}")
print(f"  calibrated auc={m_cal['auc']:.3f} brier={m_cal['brier']:.3f} ll={m_cal['log_loss']:.3f}")
use_calibrated = m_cal["brier"] <= m_raw["brier"]
print(f"  -> serving {'CALIBRATED' if use_calibrated else 'RAW'} "
      f"(lower Brier wins on holdout)")

# ============================================================================
# 6. Threshold: leak-free trainval OOF (TimeSeriesSplit) -> Youden's J
#    Report holdout @0.5 AND @tuned side-by-side (do NOT silently replace).
# ============================================================================
oof = ts_oof_proba(winner_fitted, Xtv, ytv, tscv)
_m = ~np.isnan(oof)                       # drop the unscored warm-up fold
oof_m, y_oof = oof[_m], ytv.values[_m]
pos, neg = y_oof == 1, y_oof == 0
n_pos, n_neg = max(pos.sum(), 1), max(neg.sum(), 1)


def youden_j(t):
    pred = oof_m >= t
    return (pred & pos).sum() / n_pos - (pred & neg).sum() / n_neg


ths = np.linspace(0.30, 0.70, 41)
thr_opt = float(ths[int(np.argmax([youden_j(t) for t in ths]))])
serve = p_cal if use_calibrated else p_raw
m_05 = evaluate(yte, serve, thr=0.5)
m_opt = evaluate(yte, serve, thr=thr_opt)
print("\n" + "=" * 70 + "\nDUAL-THRESHOLD HOLDOUT (winner served)\n" + "=" * 70)
print(f"  thr=0.50      acc={m_05['accuracy']:.3f} bal_acc={m_05['bal_acc']:.3f} f1={m_05['f1']:.3f}")
print(f"  thr={thr_opt:.2f}(OOF-J) acc={m_opt['accuracy']:.3f} bal_acc={m_opt['bal_acc']:.3f} f1={m_opt['f1']:.3f}")

# ============================================================================
# 7. Plots
# ============================================================================
print("\n" + "=" * 70 + "\nPLOTS\n" + "=" * 70)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

allres = pd.concat([
    algo_df.assign(family="default")[["model", "auc", "family"]],
    final_df.assign(family="tuned")[["model", "auc", "family"]],
], ignore_index=True).sort_values("auc")
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(allres["model"] + " (" + allres["family"] + ")", allres["auc"])
ax.axvline(0.5, color="gray", ls="--", label="random")
ax.axvline(train[TARGET].mean(), color="red", ls=":",
           label=f"HFA prior={train[TARGET].mean():.3f}")
ax.set_xlabel("AUC (holdout test)")
ax.set_title("CPBL 2024 home-win — algorithm comparison")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(FIG / "model_comparison.png", dpi=120)
plt.close(fig)

# THE report centrepiece: per-group N=47 holdout AUC vs leak-free
# season-OOF AUC. The gap (esp. m6 pitching .689 -> .463) is the whole
# argument for why small-holdout ranking is dangerous and why the
# season-OOF + CI machinery exists. The visual IS the conclusion.
abl_t = ablation[ablation.split == "test"].set_index("model")
order = [m for m in ["m1", "m2", "m3", "m4", "m5", "m6", "m7"]
         if m in abl_t.index]
lbl = {"m1": "m1 intercept", "m2": "m2 stadium", "m3": "m3 weather",
       "m4": "m4 team-str", "m5": "m5 batter", "m6": "m6 PITCHING",
       "m7": "m7 full"}
hold = [abl_t.loc[m, "auc"] for m in order]
oofv = [ablation_oof.get(m, np.nan) for m in order]
oofv = [0.5 if (v != v) else v for v in oofv]      # m1 NaN -> chance
x = np.arange(len(order))
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - 0.2, hold, 0.38, label="N=47 holdout AUC (noisy)",
       color="#d98", edgecolor="k", linewidth=.4)
ax.bar(x + 0.2, oofv, 0.38, label="season-OOF AUC (robust, ~455g)",
       color="#48a", edgecolor="k", linewidth=.4)
ax.axhline(0.5, color="gray", ls="--", lw=1, label="chance / HFA")
for i, (h, o) in enumerate(zip(hold, oofv)):
    ax.text(i - 0.2, h + .008, f"{h:.2f}", ha="center", fontsize=8)
    ax.text(i + 0.2, o + .008, f"{o:.2f}", ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels([lbl[m] for m in order], rotation=20, ha="right")
ax.set_ylim(0.40, max(hold) + .06)
ax.set_ylabel("AUC")
ax.set_title("Why small-holdout ranking lies: m6 pitching .689 holdout "
             "→ .46 walk-forward\n(every group collapses to ~.50 OOF — "
             "no pre-game signal at N=678)")
ax.legend(loc="upper left", fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "ablation_holdout_vs_oof.png", dpi=120)
plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 6))
for name, p in preds_for_calib.items():
    fp, mp = calibration_curve(yte, p, n_bins=8, strategy="quantile")
    ax.plot(mp, fp, marker="o", label=name)
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="perfect")
ax.set_xlabel("Predicted probability")
ax.set_ylabel("Observed home-win frequency")
ax.set_title("Calibration (holdout test)")
ax.legend(loc="best", fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "calibration.png", dpi=120)
plt.close(fig)

try:
    import shap
    clf = winner_fitted.named_steps["clf"]
    pre = winner_fitted.named_steps["pre"]
    Xtt = pre.transform(Xte)
    names = list(pre.get_feature_names_out())
    STAD_ASCII = {"樂天桃園": "Taoyuan", "洲際": "Taichung", "天母": "Tianmu",
                  "新莊": "Xinzhuang", "澄清湖": "Chengqing", "臺南": "Tainan",
                  "大巨蛋": "Dome", "其他": "Other"}
    for i, n in enumerate(names):
        for cjk, eng in STAD_ASCII.items():
            n = n.replace(cjk, eng)
        names[i] = n.replace("num__", "").replace("cat__", "")
    if isinstance(clf, (xgb.XGBClassifier, lgb.LGBMClassifier,
                        RandomForestClassifier)):
        sv = shap.TreeExplainer(clf).shap_values(Xtt)
        if isinstance(sv, list):
            sv = sv[1]
        elif hasattr(sv, "ndim") and sv.ndim == 3:
            sv = sv[:, :, 1]
        shap.summary_plot(sv, Xtt, feature_names=names, show=False,
                          max_display=15)
        plt.tight_layout()
        plt.savefig(FIG / "shap_summary.png", dpi=120, bbox_inches="tight")
        plt.close()
        print(f"  SHAP saved ({type(clf).__name__})")
    else:
        print(f"  SHAP skipped (linear winner {type(clf).__name__})")
except Exception as e:                                   # noqa: BLE001
    print(f"  SHAP skipped: {type(e).__name__}: {e}")

# ============================================================================
# 8. Shiny artifacts — precompute contract (decided up-front, not in step6)
#    R Shiny just RENDERS these; no reticulate, deploy-safe.
# ============================================================================
print("\n" + "=" * 70 + "\nSHINY ARTIFACTS\n" + "=" * 70)

# 8a. leak-free per-game OOF probabilities for the WHOLE post-warmup season
#     (each game scored by a model fit only on chronologically earlier games)
if use_calibrated:
    oof_est = CalibratedClassifierCV(clone(winner_fitted), method="isotonic",
                                     cv=TimeSeriesSplit(3))
else:
    oof_est = clone(winner_fitted)
oof_all = ts_oof_proba(oof_est, df[m7], df[TARGET], tscv)
keep = ~np.isnan(oof_all)                  # earliest fold is never scored
oofk = oof_all[keep]
pred_cols = ["game_id", "date", "stadium", "home_team", "away_team"]
predictions = df.loc[keep, pred_cols].copy()
predictions["y_true"] = df.loc[keep, TARGET].values
predictions["p_home_win"] = oofk
predictions["pred_at_0.5"] = (oofk >= 0.5).astype(int)
predictions[f"pred_at_{thr_opt:.2f}"] = (oofk >= thr_opt).astype(int)
predictions["is_holdout"] = (df.loc[keep, "date"] >= "2024-09-16").astype(int)
predictions.to_csv(EVAL / "predictions.csv", index=False)
oof_auc = (roc_auc_score(df.loc[keep, TARGET], oofk)
           if df.loc[keep, TARGET].nunique() > 1 else float("nan"))
print(f"  predictions.csv  rows={len(predictions)}/{len(df)} "
      f"(walk-forward OOF; warm-up fold unscored)  "
      f"season-OOF-AUC={oof_auc:.3f}")

# 8b. production model: serve trained on ALL post-warmup data
prod = CalibratedClassifierCV(clone(winner_fitted), method="isotonic",
                              cv=TimeSeriesSplit(3)) if use_calibrated \
    else clone(winner_fitted)
prod.fit(df[m7], df[TARGET])
joblib.dump({"model": prod, "features": m7,
             "categorical": STADIUM_CAT,
             "numeric": [c for c in m7 if c not in STADIUM_CAT],
             "threshold": thr_opt, "winner": winner_name},
            MODELS / "best_model.joblib")
print(f"  best_model.joblib  ({winner_name}, "
      f"{'isotonic-calibrated' if use_calibrated else 'raw'})")

# 8c. Shiny input contract
schema = {
    "winner": winner_name,
    "calibrated": bool(use_calibrated),
    "winner_params": winner_search.best_params_,
    "threshold_opt": thr_opt,
    "home_win_base_rate": float(trainval[TARGET].mean()),
    "n": {"train": len(train), "valid": len(valid), "test": len(test),
          "post_warmup": len(df)},
    "feature_groups": {
        "stadium": STADIUM_ALL, "weather": WEATHER_COLS,
        "team_strength": TEAM_STRENGTH, "batter_state": BATTER_STATE,
        "pitching": PITCHING},
    "model_features": m7,
    "categorical_features": STADIUM_CAT,
    "stadium_levels": sorted(df["stadium"].dropna().unique().tolist()),
    "metrics": {
        "cv_auc": cv_auc, "season_oof_auc": float(oof_auc),
        "holdout_auc": float(m_raw["auc"]),
        "holdout_auc_ci95": auc_bootstrap_ci(yte, p_raw),
        "holdout_brier_raw": float(m_raw["brier"]),
        "holdout_brier_cal": float(m_cal["brier"])},
}
(EVAL / "feature_schema.json").write_text(
    json.dumps(schema, indent=2, ensure_ascii=False, default=str),
    encoding="utf-8")
print(f"  feature_schema.json  ({len(m7)} features, "
      f"{len(schema['stadium_levels'])} stadium levels)")

# ============================================================================
# 9. Final metrics JSON
# ============================================================================
(EVAL / "_final_metrics.json").write_text(json.dumps({
    "winner": winner_name,
    "served": "calibrated" if use_calibrated else "raw",
    "cv_auc": cv_auc,
    "holdout_raw": m_raw, "holdout_calibrated": m_cal,
    "holdout_auc_ci95": auc_bootstrap_ci(yte, p_raw),
    "threshold_opt": thr_opt,
    "holdout_at_0.5": m_05, "holdout_at_opt": m_opt,
    "season_oof_auc": float(oof_auc),
    "ablation_season_oof": {k: (None if (v != v) else float(v))
                            for k, v in ablation_oof.items()},
    "ablation_season_oof_folds": ablation_oof_folds,
    "best_params": {k: v.best_params_ for k, v in searches.items()},
}, indent=2, default=str), encoding="utf-8")
print(f"\nfinal_metrics: {EVAL / '_final_metrics.json'}\nDONE.")

In [ ]:
# === Cell 9 — 結果：metrics + 4 張圖 + 投手特徵 sanity ===
import json, pathlib
from IPython.display import Image, display
p = pathlib.Path("Results/eval/_final_metrics.json")
if p.exists():
    d = json.loads(p.read_text())
    print(json.dumps(d, indent=2, ensure_ascii=False))
    print("\n>>> HEADLINE: ablation_season_oof =",
          d.get("ablation_season_oof"))
    print(">>> m6 per-fold =",
          d.get("ablation_season_oof_folds",{}).get("m6"))
else:
    print("❌ _final_metrics.json 沒生成 — 看上方 step3 紅字")
# pitcher features must have reached the outputs
mr = pathlib.Path("data/processed/model_ready_data.csv")
hdr = mr.read_text(encoding="utf-8").splitlines()[0].split(",") if mr.exists() else []
print("\npitcher cols in model_ready:",
      "OK" if "diff_spERA_l5" in hdr else "❌ MISSING")
for f in ["ablation_holdout_vs_oof.png","model_comparison.png",
          "calibration.png","shap_summary.png"]:
    fp = f"Results/figures/{f}"
    if pathlib.Path(fp).exists():
        print("\n"+fp); display(Image(fp))
pc = pathlib.Path("Results/eval/predictions.csv")
print("\npredictions.csv:", "OK" if pc.exists() else "MISSING")

### Cell 10（選用）打包 / 推回 artifacts

只想下載結果 → 跑前半（zip）。要把 `Results/eval/*` 推回 GitHub 給之後
Shiny 用 → 填 PAT 跑後半（會 shallow-clone 一份只為提交 artifacts）。

In [ ]:
# === Cell 10（選用）打包 + 可選推回 ===
import shutil, subprocess
shutil.make_archive("cpbl_artifacts","zip",".",
                    base_dir="Results")
print("→ cpbl_artifacts.zip 可從左側檔案面板下載（含 Results/eval + figures）")

PUSH = False                    # 改 True 並填 PAT 才會推回
if PUSH:
    from getpass import getpass
    tok = getpass("GitHub PAT (repo scope): ")
    REPO = "https://github.com/jiangjiangian/data_science_final_project.git"
    BR   = "claude/setup-main-agent-BhYTE"
    subprocess.run(["rm","-rf","_pushrepo"], check=False)
    subprocess.run(["git","clone","--depth","1","--branch",BR,
                    REPO,"_pushrepo"], check=True)
    subprocess.run(["cp","-r","Results/eval","Results/figures",
                    "_pushrepo/Results/"], check=True)
    subprocess.run(["git","-C","_pushrepo","add","-f",
                    "Results/eval","Results/figures"], check=True)
    subprocess.run(["git","-C","_pushrepo","-c","user.email=colab@run",
                    "-c","user.name=colab","commit","-m",
                    "data: pipeline artifacts (single-notebook run)"],
                   check=False)
    url = REPO.replace("https://", f"https://{tok}@")
    r = subprocess.run(["git","-C","_pushrepo","push",url,f"HEAD:{BR}"],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr or "pushed ✓")